In [5]:
import pandas as pd
import numpy as np

In [10]:
analyst = pd.read_pickle("../GPT 3.5/Weights/analyst.pkl")
finbert = pd.read_pickle("../GPT 3.5/Weights/finbert.pkl")
llm_s = pd.read_pickle("../GPT 3.5/Weights/llm-s.pkl")

In [21]:
def calculate_weight_correlation(weights_list1, weights_list2):
    """
    Calculate the cross-sectional correlation of portfolio weights over time.
    
    weights_list1, weights_list2: Lists of dictionaries, e.g., [{permno: weight, ...}, ...]
    """
    correlations = []
    
    for w1, w2 in zip(weights_list1, weights_list2):
        # Union of all unique assets in both portfolios for this time period
        all_assets = set(w1.keys()).union(set(w2.keys()))
        
        # If both portfolios are completely liquidated (empty)
        if len(all_assets) == 0:
            correlations.append(1.0) 
            continue
            
        # Align weights into arrays (default to 0.0 if asset is missing)
        arr1 = np.array([w1.get(asset, 0.0) for asset in all_assets])
        arr2 = np.array([w2.get(asset, 0.0) for asset in all_assets])
        
        # Pearson correlation requires variance > 0. 
        # If one portfolio is liquidated (all 0s), its standard deviation is 0.
        if np.std(arr1) == 0 or np.std(arr2) == 0:
            if np.array_equal(arr1, arr2):
                correlations.append(1.0) # Both have identical constant weights (e.g., all 0s)
            else:
                correlations.append(0.0) # One is liquidated, the other is not
            continue
            
        # Calculate Pearson correlation coefficient
        corr = np.dot(arr1, arr2) / (np.linalg.norm(arr1) * np.linalg.norm(arr2)) #np.corrcoef(arr1, arr2)[0, 1]
        correlations.append(corr)
        
    avg_correlation = np.mean(correlations)
    
    return correlations, avg_correlation

In [22]:
# Example usage assuming you extracted 'portfolio_weights' lists from two results dataframes:
analyst_list = analyst.tolist()
finbert_list = finbert.tolist()
llm_s_list = llm_s.tolist()
corr_history, avg_corr = calculate_weight_correlation(analyst_list, llm_s_list)
print(f"Time-series Average Weight Correlation: {avg_corr:.4f}")

Time-series Average Weight Correlation: 0.0408


In [23]:
corr_history, avg_corr = calculate_weight_correlation(llm_s_list, finbert_list)
print(f"Time-series Average Weight Correlation: {avg_corr:.4f}")

Time-series Average Weight Correlation: 0.0198
